# Multi Anonymization Runner


## Notebook Guide
- Adjust the configuration cells below (mode, parameter grids, optional variants).
- Run each command-definition cell to register tasks (they default to dry-run previews).
- Use the pipeline control cell at the bottom to execute selected steps or the full sequence.


## Imports & Typing Helpers


In [91]:

import os
import sys
import subprocess
from dataclasses import dataclass
from pathlib import Path
from shutil import which
from typing import Dict, Optional, Sequence


## Command Helpers


In [92]:

TEAM_IDS_PREP = tuple(i for i in range(1, 23) if i != 21)
TEAM_IDS_CONTEST = tuple(i for i in range(1, 25) if i != 21)

DEFAULT_TEAM_IDS: Sequence[int] = TEAM_IDS_PREP
DEFAULT_VARIANTS: Sequence[str] = ("3",)
TEAM_22_ONLY = (22,)


@dataclass
class CommandSpec:
    key: str
    label: str
    template: Sequence[str]
    category: str
    description: str = ""
    cwd: Optional[str] = None
    continue_on_error: bool = False
    variants: Optional[Sequence[str]] = None
    strict: bool = True


COMMAND_REGISTRY: Dict[str, CommandSpec] = {}
PIPELINE_ORDER: list[str] = []


def slugify(value: object) -> str:
    text = str(value).replace("-", "m").replace(".", "p")
    return text


def execute_command(
    template: Sequence[str],
    *,
    teams: Optional[Sequence[int]] = None,
    variants: Optional[Sequence[str]] = None,
    dry_run: bool = False,
    strict: bool = True,
    continue_on_error: bool = False,
    cwd: Optional[str] = None,
) -> None:
    if not template:
        raise ValueError("Template must contain at least one argument.")

    id_placeholder = any(
        isinstance(arg, str) and "{id" in arg for arg in template
    )
    variant_placeholder = any(
        isinstance(arg, str) and "{variant" in arg for arg in template
    )

    if strict and not (id_placeholder or variant_placeholder):
        raise ValueError("No {id} or {variant} placeholder found in template.")

    base_cmd = list(template)
    if isinstance(base_cmd[0], str) and base_cmd[0] in ("python", "python3"):
        base_cmd[0] = sys.executable

    exe = base_cmd[0]
    if isinstance(exe, str) and os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    team_iter = (
        tuple(DEFAULT_TEAM_IDS if teams is None else teams)
        if id_placeholder
        else (None,)
    )
    variant_iter = (
        tuple(DEFAULT_VARIANTS if variants is None else variants)
        if variant_placeholder
        else (None,)
    )

    if strict and variant_placeholder and not variant_iter:
        raise ValueError("Variant placeholders require at least one variant.")

    for team in team_iter:
        for variant in variant_iter:
            format_kwargs = {}
            if team is not None:
                format_kwargs["id"] = team
            if variant is not None:
                format_kwargs["variant"] = variant

            cmd = [
                arg.format(**format_kwargs) if isinstance(arg, str) else arg
                for arg in base_cmd
            ]

            def is_output_argument(index: int) -> bool:
                if index == 0:
                    return False
                prev = cmd[index - 1]
                return isinstance(prev, str) and (
                    prev in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                    or prev.startswith("--out")
                )

            missing_inputs = []
            for index, arg in enumerate(cmd):
                if isinstance(arg, str) and arg.lower().endswith((".csv", ".json")) and not is_output_argument(index):
                    candidate = arg if cwd is None else os.path.join(cwd, arg)
                    if not os.path.exists(candidate):
                        missing_inputs.append(arg)

            print(">>", " ".join(str(part) for part in cmd))
            if missing_inputs:
                scope_bits = []
                if team is not None:
                    scope_bits.append(f"team {team}")
                if variant is not None:
                    scope_bits.append(f"variant {variant}")
                scope = " ".join(scope_bits) or "command"
                message = f"[{scope}] Missing input files: {missing_inputs}"
                if continue_on_error:
                    print("!!", message)
                    continue
                raise FileNotFoundError(message)

            if dry_run:
                continue

            try:
                completed = subprocess.run(
                    cmd,
                    check=True,
                    cwd=cwd,
                    capture_output=True,
                    text=True,
                )
                if completed.stdout:
                    print(completed.stdout.strip())
            except subprocess.CalledProcessError as exc:
                print(f"[ERROR] execution failed with code {exc.returncode}")
                if exc.stdout:
                    print("--- stdout ---")
                    print(exc.stdout.strip())
                if exc.stderr:
                    print("--- stderr ---")
                    print(exc.stderr.strip())
                if not continue_on_error:
                    raise


def register_command(
    key: str,
    *,
    label: str,
    template: Sequence[str],
    category: str,
    description: str = "",
    cwd: Optional[str] = None,
    continue_on_error: bool = False,
    variants: Optional[Sequence[str]] = None,
    strict: bool = True,
) -> CommandSpec:
    spec = CommandSpec(
        key=key,
        label=label,
        template=list(template),
        category=category,
        description=description,
        cwd=cwd,
        continue_on_error=continue_on_error,
        variants=None if variants is None else tuple(variants),
        strict=strict,
    )
    COMMAND_REGISTRY[key] = spec
    if key not in PIPELINE_ORDER:
        PIPELINE_ORDER.append(key)
    return spec


def run_command(
    *,
    key: str,
    label: str,
    category: str,
    template: Sequence[str],
    description: str = "",
    cwd: Optional[str] = None,
    continue_on_error: bool = False,
    variants: Optional[Sequence[str]] = None,
    teams: Optional[Sequence[int]] = None,
    dry_run: bool = False,
    strict: bool = True,
) -> None:
    spec = register_command(
        key,
        label=label,
        template=template,
        category=category,
        description=description,
        cwd=cwd,
        continue_on_error=continue_on_error,
        variants=variants,
        strict=strict,
    )
    print(f"=== {label} ({key}) ===")
    execute_command(
        spec.template,
        teams=teams,
        variants=variants if variants is not None else spec.variants,
        dry_run=dry_run,
        strict=strict,
        continue_on_error=continue_on_error,
        cwd=cwd,
    )


def rerun_command(
    key: str,
    *,
    teams: Optional[Sequence[int]] = None,
    variants: Optional[Sequence[str]] = None,
    dry_run: bool = False,
    continue_on_error: Optional[bool] = None,
    cwd: Optional[str] = None,
) -> None:
    if key not in COMMAND_REGISTRY:
        raise KeyError(f"Unknown command key: {key}")
    spec = COMMAND_REGISTRY[key]
    effective_continue = (
        spec.continue_on_error if continue_on_error is None else continue_on_error
    )
    print(f"=== {spec.label} ({key}) ===")
    execute_command(
        spec.template,
        teams=teams,
        variants=variants if variants is not None else spec.variants,
        dry_run=dry_run,
        strict=spec.strict,
        continue_on_error=effective_continue,
        cwd=cwd if cwd is not None else spec.cwd,
    )


def list_commands(category: Optional[str] = None) -> None:
    for key in PIPELINE_ORDER:
        spec = COMMAND_REGISTRY.get(key)
        if spec is None:
            continue
        if category and spec.category != category:
            continue
        print(f"{key:>24}  {spec.label} [{spec.category}]")
        if spec.description:
            print(f"    {spec.description}")


def run_pipeline(
    keys: Optional[Sequence[str]] = None,
    *,
    teams: Optional[Sequence[int]] = None,
    variants: Optional[Sequence[str]] = None,
    dry_run: bool = False,
    continue_on_error: Optional[bool] = None,
) -> None:
    sequence = keys if keys is not None else PIPELINE_ORDER
    for key in sequence:
        rerun_command(
            key,
            teams=teams,
            variants=variants,
            dry_run=dry_run,
            continue_on_error=continue_on_error,
        )


## Environment Setup


In [93]:
PROJECT_ROOT_OVERRIDE: Optional[str] = None

def find_project_root(start: Path) -> Path:
    markers = ("requirements.txt", "GUIDE_FOR_BEGINNERS.md")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return start


PROJECT_ROOT = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE
    else find_project_root(Path.cwd())
)
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {PROJECT_ROOT}")


Working directory set to: /home/kikuchih/pwscup2025-scripts


## Mode Configuration


In [94]:
MODE_PRESETS = {
    "prep": {
        "label": "Preliminary datasets (B/C/D)",
        "original_prefix": "B",
        "anon_prefix": "C",
        "model_prefix": "D",
        "all_dir": "PWSCUP2025_Pre_Data_for_Attack",
        "all_prefix": "A",
        "input_dir": "in",
        "anon_dir": "out_anonymized",
        "eval_dir": "out_eval",
        "model_dir": "out_anonymized",
        "teams": TEAM_22_ONLY,
        "variants": ("3",),
    },
    "contest": {
        "label": "Contest datasets (BB/CC/DD)",
        "original_prefix": "BB",
        "anon_prefix": "CC",
        "model_prefix": "DD",
        "all_dir": "PWSCUP2025_Pre_Data_for_Attack",
        "all_prefix": "AA",
        "input_dir": "in",
        "anon_dir": "out_anonymized",
        "eval_dir": "out_eval",
        "model_dir": "out_anonymized",
        "teams": TEAM_IDS_CONTEST,
        "variants": ("1", "2", "3"),
    },
}

MODE = "prep"  # change as needed

mode_config = MODE_PRESETS[MODE]
mode_label = mode_config["label"]
mode_original = mode_config["original_prefix"]
mode_anon = mode_config["anon_prefix"]
mode_model = mode_config["model_prefix"]
mode_all_dir = mode_config.get("all_dir")
mode_all_prefix = mode_config.get("all_prefix")
input_root = mode_config["input_dir"]
anon_root = mode_config["anon_dir"]
eval_root = mode_config["eval_dir"]
model_root = mode_config["model_dir"]

DEFAULT_TEAM_IDS = mode_config["teams"]
DEFAULT_VARIANTS = mode_config["variants"]

print(f"Mode: {MODE} ({mode_label})")
print(f"Teams: {DEFAULT_TEAM_IDS}")
print(f"Variants: {DEFAULT_VARIANTS}")


Mode: prep (Preliminary datasets (B/C/D))
Teams: (22,)
Variants: ('3',)


## Directory Preparation


In [95]:

folders_to_create = {input_root, anon_root, eval_root, model_root}
for folder in folders_to_create:
    Path(folder).mkdir(parents=True, exist_ok=True)
print("Ensured directories:", sorted(folders_to_create))


Ensured directories: ['in', 'out_anonymized', 'out_eval']


## Parameter Grids


In [103]:

BASE_RANDOM_SEED = 42
MONDRIAN_METHOD = "oka"
MONDRIAN_K_VALUES = [11, 15, 20, 30, 50]

DATASYNTH_NUM_TUPLES = 10000
DATASYNTH_SEED = 42
DATASYNTH_RECIPES = [
    {
        "key_prefix": "privbayes",
        "label": "DataSynthesizer PrivBayes",
        "mode": "correlated_attribute_mode",
        "epsilons": [0.1, 0.5, 1.0, 3, 5, 10, 15, 20, 25, 30, 40, 50],
        "k_values": [3],
    },
    {
        "key_prefix": "independent",
        "label": "DataSynthesizer Independent",
        "mode": "independent_attribute_mode",
        "epsilons": [0.1, 0.5, 1.0, 3, 5, 10, 15, 20, 25, 30, 40, 50],
        "k_values": [3],
    },
]


## Preprocessing (Aデータ自動修正)
Aデータ（例: A01.csv）の数値範囲外や欠損値を `util/check_and_fix_csv.py` で補正します。

In [97]:
from pathlib import Path
import subprocess
import sys

a_prefix_dir = mode_all_dir
a_prefix_code = mode_all_prefix
a_team_ids = tuple(TEAM_IDS_PREP)
columns_range_path = Path('data/pre_columns_range.json')

if not a_prefix_dir or not a_prefix_code:
    print('mode_all_dir / mode_all_prefix が未設定のためスキップします。')
elif not columns_range_path.exists():
    raise FileNotFoundError(columns_range_path)
else:
    for team in a_team_ids:
        src = Path(f'{input_root}/{a_prefix_dir}/{a_prefix_code}{team:02d}.csv')
        dst = src.with_name(src.stem + '_fix.csv')
        if not src.exists():
            print(f'[SKIP] {src} not found.')
            continue
        if dst.exists():
            print(f'[OK] {dst} already exists.')
            continue
        print(f'[PREPROCESS] Fixing {src} -> {dst}')
        result = subprocess.run(
            [
                sys.executable,
                'util/check_and_fix_csv.py',
                str(src),
                str(columns_range_path),
                str(dst),
            ],
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout.strip())
        if result.returncode != 0:
            if result.stderr:
                print(result.stderr.strip())
            raise RuntimeError(f'Failed to preprocess {src}')


[OK] in/PWSCUP2025_Pre_Data_for_Attack/A01_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A02_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A03_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A04_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A05_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A06_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A07_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A08_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A09_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A10_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A11_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A12_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A13_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A14_fix.csv already exists.
[OK] in/PWSCUP2025_Pre_Data_for_Attack/A15_fix.csv already exi

## Baseline Ci Anonymization


In [82]:

run_command(
    key="ci_baseline",
    label="Ci anonymization (ano_fixed.py)",
    category="baseline",
    description="Generate anonymized CSVs from original data.",
    template=[
        "python",
        "anonymization/ano_fixed.py",
        f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
        "-o",
        f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}.csv",
        "--seed",
        str(BASE_RANDOM_SEED),
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="ci_eval",
    label="Evaluate baseline anonymization",
    category="evaluation",
    description="Score baseline anonymized CSVs against the originals.",
    template=[
        "python",
        "evaluation/eval_all_fixed.py",
        f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
        f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}.csv",
        "-o",
        f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_eval.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="ci_eval_csv",
    label="Convert baseline eval reports",
    category="evaluation",
    description="Turn baseline evaluation TXT files into CSV summaries.",
    template=[
        "python",
        "anonymization/evaltxt_to_csv.py",
        f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_eval.txt",
        "-o",
        f"{eval_root}/{mode_anon}{{id:02d}}_eval.csv",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
    strict=False,
    continue_on_error=True,
)


=== Ci anonymization (ano_fixed.py) (ci_baseline) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/B22_3.csv -o out_anonymized/C22_3.csv --seed 42
=== Evaluate baseline anonymization (ci_eval) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3.csv -o out_eval/C22_3_eval.txt
=== Convert baseline eval reports (ci_eval_csv) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_eval.txt -o out_eval/C22_eval.csv


## Di Model Training


In [98]:
run_command(
    key="di_train_bi",
    label="Train Di (Bi only)",
    category="di",
    description="Train xgbt models using only the original Bi data.",
    template=[
        "python",
        "analysis/xgbt_train_fixed.py",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        "-o",
        f"{model_root}/{mode_model}{id:02d}_Bi_{variant}.json",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_eval_bi",
    label="Evaluate Di (Bi only)",
    category="di_eval",
    description="Evaluate Bi-trained Di models against the original Bi CSV using MLacc.py.",
    template=[
        "python",
        "anonymization/MLacc.py",
        f"{model_root}/{mode_model}{id:02d}_Bi_{variant}.json",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        "--accuracy-out",
        f"{model_root}/{mode_model}{id:02d}_Bi_{variant}.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_train_ci",
    label="Train Di (Ci only)",
    category="di",
    description="Train xgbt models using anonymized Ci CSV data.",
    template=[
        "python",
        "analysis/xgbt_train_fixed.py",
        f"{anon_root}/{mode_anon}{id:02d}_{variant}.csv",
        "-o",
        f"{model_root}/{mode_model}{id:02d}_Ci_{variant}.json",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_eval_ci",
    label="Evaluate Di (Ci only)",
    category="di_eval",
    description="Evaluate Ci-trained Di models against the anonymized Ci CSV using MLacc.py.",
    template=[
        "python",
        "anonymization/MLacc.py",
        f"{model_root}/{mode_model}{id:02d}_Ci_{variant}.json",
        f"{anon_root}/{mode_anon}{id:02d}_{variant}.csv",
        "--accuracy-out",
        f"{model_root}/{mode_model}{id:02d}_Ci_{variant}.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_train_bici",
    label="Train Di (Bi + Ci)",
    category="di",
    description="Create Di models from paired Bi and Ci datasets.",
    template=[
        "python",
        "anonymization/gen_Di_fixed.py",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        f"{anon_root}/{mode_anon}{id:02d}_{variant}.csv",
        "-o",
        f"{model_root}/{mode_model}{id:02d}_BiCi_{variant}.json",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_eval_bici",
    label="Evaluate Di (Bi + Ci)",
    category="di_eval",
    description="Evaluate Bi+Ci-trained Di models against the original Bi CSV using MLacc.py.",
    template=[
        "python",
        "anonymization/MLacc.py",
        f"{model_root}/{mode_model}{id:02d}_BiCi_{variant}.json",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        "--accuracy-out",
        f"{model_root}/{mode_model}{id:02d}_BiCi_{variant}.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_train_bici_multi",
    label="Train Di (Bi + Ci, multi)",
    category="di",
    description="Generate multi-model Di JSON via gen_Di_multi.py.",
    template=[
        "python",
        "anonymization/gen_Di_multi.py",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        f"{anon_root}/{mode_anon}{id:02d}_{variant}.csv",
        "-o",
        f"{model_root}/{mode_model}{id:02d}_BiCi_multi_{variant}.json",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

run_command(
    key="di_eval_bici_multi",
    label="Evaluate Di (Bi + Ci, multi)",
    category="di_eval",
    description="Evaluate multi-source Di models against the original Bi CSV using MLacc.py.",
    template=[
        "python",
        "anonymization/MLacc.py",
        f"{model_root}/{mode_model}{id:02d}_BiCi_multi_{variant}.json",
        f"{input_root}/{mode_original}{id:02d}_{variant}.csv",
        "--accuracy-out",
        f"{model_root}/{mode_model}{id:02d}_BiCi_multi_{variant}.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=True,
)

a_prefix_dir = mode_all_dir
a_prefix_code = mode_all_prefix
a_team_ids = tuple(TEAM_IDS_PREP)
a_csv_paths = []
missing_a_paths = []
if a_prefix_dir and a_prefix_code:
    for team in a_team_ids:
        candidate = f"{input_root}/{a_prefix_dir}/{a_prefix_code}{team:02d}_fix.csv"
        if Path(candidate).exists():
            a_csv_paths.append(candidate)
        else:
            missing_a_paths.append(candidate)

if missing_a_paths:
    print("Missing A-prefixed datasets:")
    for path in missing_a_paths:
        print(" -", path)
elif a_csv_paths:
    run_command(
        key="di_train_a_multi",
        label="Train Di from A datasets",
        category="di",
        description="Train a Di model using all A-prefixed CSVs via gen_Di_multi.py.",
        template=[
            "python",
            "anonymization/gen_Di_multi.py",
            *a_csv_paths,
            "-o",
            f"{model_root}/{mode_model}ALL_multi.json",
        ],
        dry_run=False,
        strict=False,
    )
else:
    print("No A-prefixed datasets configured; skipping aggregated Di training.")


=== Train Di (Bi only) (di_train_bi) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/B22_3.csv -o out_anonymized/D22_Bi_3.json
=== Train Di (Ci only) (di_train_ci) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py out_anonymized/C22_3.csv -o out_anonymized/D22_Ci_3.json
=== Train Di (Bi + Ci) (di_train_bici) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_fixed.py in/B22_3.csv out_anonymized/C22_3.csv -o out_anonymized/D22_BiCi_3.json
=== Train Di (Bi + Ci, multi) (di_train_bici_multi) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_multi.py in/B22_3.csv out_anonymized/C22_3.csv -o out_anonymized/D22_BiCi_multi_3.json
=== Train Di from A datasets (di_train_a_multi) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_multi.py in/PWSCUP2025_Pre_Data_for_Attack/A01_fix.csv in/PWSCUP2025_Pre_Data_for_Attack/A02_fix

## Mondrian Anonymization


In [104]:

if not MONDRIAN_K_VALUES:
    print("MONDRIAN_K_VALUES is empty; skipping Mondrian commands.")

for k in MONDRIAN_K_VALUES:
    suffix = f"mondrian_k{k}_{MONDRIAN_METHOD}"
    run_command(
        key=f"mondrian_{k:02d}",
        label=f"Mondrian anonymization (k={k})",
        category="mondrian",
        description=f"Run the custom Mondrian anonymizer with k={k}.",
        template=[
            "python",
            "third_party/k-anonymity/anonymize-pws.py",
            "--method",
            MONDRIAN_METHOD,
            "--k",
            str(k),
            "--input",
            f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
            "-o",
            f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}_{suffix}.csv",
        ],
        variants=DEFAULT_VARIANTS,
        dry_run=False,
    )

    run_command(
        key=f"mondrian_eval_{k:02d}",
        label=f"Evaluate Mondrian outputs (k={k})",
        category="mondrian",
        description="Score the Mondrian anonymized CSVs.",
        template=[
            "python",
            "evaluation/eval_all_fixed.py",
            f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
            f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}_{suffix}.csv",
            "-o",
            f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_{suffix}_eval.txt",
        ],
        variants=DEFAULT_VARIANTS,
        dry_run=False,
    )

    run_command(
        key=f"mondrian_eval_csv_{k:02d}",
        label=f"Convert Mondrian eval reports (k={k})",
        category="mondrian",
        description="Turn Mondrian evaluation TXT files into CSV summaries.",
        template=[
            "python",
            "anonymization/evaltxt_to_csv.py",
            f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_{suffix}_eval.txt",
            "-o",
            f"{eval_root}/{mode_anon}{{id:02d}}_{suffix}_eval.csv",
        ],
        variants=DEFAULT_VARIANTS,
        dry_run=False,
        strict=False,
        continue_on_error=True,
    )


=== Mondrian anonymization (k=11) (mondrian_11) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method oka --k 11 --input in/B22_3.csv -o out_anonymized/C22_3_mondrian_k11_oka.csv
Information Loss: 19671.822494257496
=== Evaluate Mondrian outputs (k=11) (mondrian_eval_11) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_mondrian_k11_oka.csv -o out_eval/C22_3_mondrian_k11_oka_eval.txt
stats_diff max_abs: 0.1445591025302826
LR_asthma_diff max_abs: 0.7538379097877401
KW_IND_diff max_abs: 0.2375026109079637
Ci utility: 54.390825484874625 / 80
=== Convert Mondrian eval reports (k=11) (mondrian_eval_csv_11) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_mondrian_k11_oka_eval.txt -o out_eval/C22_mondrian_k11_oka_eval.csv
=== Mondrian anonymization (k=15) (mondrian_15) ===
>> /home/kikuchih/miniconda3/envs/p

## DataSynthesizer Pipelines


In [47]:

if not DATASYNTH_RECIPES:
    print("No DataSynth recipes configured.")

for recipe in DATASYNTH_RECIPES:
    epsilons = list(recipe.get("epsilons", []))
    k_values = list(recipe.get("k_values", []))
    recipe_variants = recipe.get("variants")
    if not epsilons or not k_values:
        continue

    key_prefix = recipe["key_prefix"]
    label_prefix = recipe["label"]
    mode_name = recipe["mode"]

    for epsilon in epsilons:
        for k in k_values:
            suffix = f"{key_prefix}_e{slugify(epsilon)}_k{k}"
            output_base = f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}_ds_{mode_name}_e{epsilon}_k{k}"

            run_command(
                key=f"ds_{suffix}",
                label=f"{label_prefix} (epsilon={epsilon}, k={k})",
                category="datasynth",
                description=f"Generate synthetic data with mode={mode_name}, epsilon={epsilon}, k={k}.",
                template=[
                    "python",
                    "anonymization/ano_DataSynthesizer.py",
                    f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
                    "-o",
                    f"{output_base}.csv",
                    "--epsilon",
                    str(epsilon),
                    "--k",
                    str(k),
                    "--mode",
                    mode_name,
                    "--num_tuples",
                    str(DATASYNTH_NUM_TUPLES),
                    "--seed",
                    str(DATASYNTH_SEED),
                ],
                variants=recipe_variants if recipe_variants else DEFAULT_VARIANTS,
                dry_run=True,
            )

            run_command(
                key=f"ds_fix_{suffix}",
                label=f"DataFix for {label_prefix} (epsilon={epsilon}, k={k})",
                category="datasynth",
                description="Apply schema fixes to the synthetic CSV output.",
                template=[
                    "python",
                    "anonymization/datafix.py",
                    f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
                    f"{output_base}.csv",
                    "-o",
                    f"{output_base}_fixed.csv",
                ],
                variants=recipe_variants if recipe_variants else DEFAULT_VARIANTS,
                dry_run=True,
            )

            run_command(
                key=f"ds_eval_{suffix}",
                label=f"Evaluate DataSynth output (epsilon={epsilon}, k={k})",
                category="datasynth",
                description="Score the fixed synthetic CSV against the original data.",
                template=[
                    "python",
                    "evaluation/eval_all_fixed.py",
                    f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
                    f"{output_base}_fixed.csv",
                    "-o",
                    f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_ds_{mode_name}_e{epsilon}_k{k}_fixed_eval.txt",
                ],
                variants=recipe_variants if recipe_variants else DEFAULT_VARIANTS,
                dry_run=True,
            )

            run_command(
                key=f"ds_eval_csv_{suffix}",
                label=f"Convert DataSynth eval reports (epsilon={epsilon}, k={k})",
                category="datasynth",
                description="Turn DataSynth evaluation TXT reports into CSV summaries.",
                template=[
                    "python",
                    "anonymization/evaltxt_to_csv.py",
                    f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_ds_{mode_name}_e{epsilon}_k{k}_fixed_eval.txt",
                    "-o",
                    f"{eval_root}/{mode_anon}{{id:02d}}_ds_{mode_name}_fixed_eval.csv",
                ],
                variants=recipe_variants if recipe_variants else DEFAULT_VARIANTS,
                dry_run=True,
                strict=False,
                continue_on_error=True,
            )


=== DataSynthesizer PrivBayes (epsilon=0.1, k=3) (ds_privbayes_e0p1_k3) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_DataSynthesizer.py in/B22_3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3.csv --epsilon 0.1 --k 3 --mode correlated_attribute_mode --num_tuples 10000 --seed 42
=== DataFix for DataSynthesizer PrivBayes (epsilon=0.1, k=3) (ds_fix_privbayes_e0p1_k3) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/datafix.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3.csv -o out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed.csv
=== Evaluate DataSynth output (epsilon=0.1, k=3) (ds_eval_privbayes_e0p1_k3) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed.csv -o out_eval/C22_3_ds_correlated_attribute_mode_e0.1_k3_fixed_eval.txt
=== Convert DataSynth eval reports 

## synthpop Anonymization

Python synthpopパッケージを用いた匿名化データ生成パイプラインです。


In [102]:
SYNTHPOP_NUM_TUPLES = 10000  # 必要に応じて変更
SYNTHPOP_SEED = 42

run_command(
    key="synthpop",
    label="synthpop anonymization (ano_synthpop.py)",
    category="synthpop",
    description="Generate anonymized CSVs using synthpop.",
    template=[
        "python",
        "anonymization/ano_synthpop.py",
        f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
        "-o",
        f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}_synthpop.csv",
        "--seed",
        str(SYNTHPOP_SEED),
        "--num_tuples",
        str(SYNTHPOP_NUM_TUPLES),
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=False,
)

run_command(
    key="synthpop_eval",
    label="Evaluate synthpop anonymization",
    category="synthpop",
    description="Score synthpop-anonymized CSVs against the originals.",
    template=[
        "python",
        "evaluation/eval_all_fixed.py",
        f"{input_root}/{mode_original}{{id:02d}}_{{variant}}.csv",
        f"{anon_root}/{mode_anon}{{id:02d}}_{{variant}}_synthpop.csv",
        "-o",
        f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_synthpop_eval.txt",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=False,
)

run_command(
    key="synthpop_eval_csv",
    label="Convert synthpop eval reports",
    category="synthpop",
    description="Turn synthpop evaluation TXT files into CSV summaries.",
    template=[
        "python",
        "anonymization/evaltxt_to_csv.py",
        f"{eval_root}/{mode_anon}{{id:02d}}_{{variant}}_synthpop_eval.txt",
        "-o",
        f"{eval_root}/{mode_anon}{{id:02d}}_synthpop_eval.csv",
    ],
    variants=DEFAULT_VARIANTS,
    dry_run=False,
    strict=False,
    continue_on_error=True,
)


=== synthpop anonymization (ano_synthpop.py) (synthpop) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_synthpop.py in/B22_3.csv -o out_anonymized/C22_3_synthpop.csv --seed 42 --num_tuples 10000
Saved synthpop-anonymized CSV: out_anonymized/C22_3_synthpop.csv
=== Evaluate synthpop anonymization (synthpop_eval) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3_synthpop.csv -o out_eval/C22_3_synthpop_eval.txt
stats_diff max_abs: 0.3271815572788302
LR_asthma_diff max_abs: 0.8669413725829306
KW_IND_diff max_abs: 0.216242960530939
Ci utility: 45.2490510465694 / 80
=== Convert synthpop eval reports (synthpop_eval_csv) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_synthpop_eval.txt -o out_eval/C22_synthpop_eval.csv


## Pipeline Control


In [50]:

RUN_PIPELINE = True
DRY_RUN = False
TARGET_TEAMS = None      # e.g., TEAM_22_ONLY or (1, 2, 3)
TARGET_VARIANTS = None   # e.g., ("3",)
SELECTED_KEYS = None     # e.g., ("ci_baseline", "ci_eval")

if RUN_PIPELINE:
    run_pipeline(
        keys=SELECTED_KEYS,
        teams=TARGET_TEAMS,
        variants=TARGET_VARIANTS,
        dry_run=DRY_RUN,
    )
else:
    print("Pipeline execution skipped (set RUN_PIPELINE = True to run).")


=== Ci anonymization (ano_fixed.py) (ci_baseline) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/B22_3.csv -o out_anonymized/C22_3.csv --seed 42
=== Evaluate baseline anonymization (ci_eval) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python evaluation/eval_all_fixed.py in/B22_3.csv out_anonymized/C22_3.csv -o out_eval/C22_3_eval.txt
stats_diff max_abs: 0.2162671256181917
LR_asthma_diff max_abs: 0.9954029273019501
KW_IND_diff max_abs: 0.1534746161192847
Ci utility: 48.371764106847635 / 80
=== Convert baseline eval reports (ci_eval_csv) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/evaltxt_to_csv.py out_eval/C22_3_eval.txt -o out_eval/C22_eval.csv
=== Train Di (Bi only) (di_train_bi) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/B22_3.csv -o out_anonymized/D22_Bi_3.json
Validation Accuracy (threshold=0.5): 0.888000
Saved model JSON to: out_anonymized/D22_Bi_3.jso

KeyboardInterrupt: 